# Backfill historical USGS measurements example

1. Make the Data Acquisition -> USGS Measurement Group is populated with sites you want data for and there is a USGS number in the Agency Aliases->USGS Station Number location group. See populate_USGS_measurement_group.ipnb.

2. Make sure you have the overwrite attrubite set to 0 if you don't want the USGS NWIS measurements to overwrite existing measurements in your database where there is conflicting information (e.g. same measurement number or similar collection time)


In [1]:
from datetime import datetime, timezone
import pandas as pd
from dataretrieval import nwis
import pytz
import math
import numpy as np
import cwms
import json

# load .env python environment for storing API_KEY
# .env file can be stored a parent directory of script
from dotenv import load_dotenv
import os


In [2]:
# grab API variables from .env file
load_dotenv()
APIROOT = os.getenv("API_ROOT")
OFFICE = os.getenv("OFFICE")
APIKEY = os.getenv('API_KEY')


In [3]:
# api key needs to have the test "apikey" as a prefix to work with CDA 
apiKey = "apikey " + APIKEY
api = cwms.api.init_session(api_root = APIROOT, api_key = apiKey)

In [4]:
# get Agency Aliases -> USGS Station Number and merge with Data Acquisition -> USGS Measurements
usgs_alias_group = cwms.get_location_group(loc_group_id="USGS Station Number",
                                 category_id="Agency Aliases",
                                 office_id="CWMS")

usgs_measurement_locs = cwms.get_location_group(loc_group_id='USGS Measurements',
                                 category_id='Data Acquisition',
                                 office_id=OFFICE)

# merge them together
measurement_site_df = pd.merge(usgs_measurement_locs.df, usgs_alias_group.df, on='location-id', how='inner', left_on=None, right_on=None)
# drop any that don't have a USGS id
measurement_site_df=measurement_site_df[measurement_site_df['alias-id'].notnull()]


In [5]:
# if you want to backfill for all sites in the Data Acquisition -> USGS Measurement Group, you would put all the code below into this loop
for index, row in measurement_site_df.iterrows():
    cwms_loc = row['location-id']
    usgs_site = row['alias-id']
    print(cwms_loc, usgs_site)


GrandRapids 05211000
Taconite 05212700
Baraboo 05405000
Stratford 05399500
RibFalls 05396000
BlackRivFalls 053813595
Neilsville 05381000
Northfield 05355024
FortSnelling 05330920
Bruce 05356500
ChippewaFalls 05365500
Kelly 05397500
Sheldon 05362000
Sandstone 05336700
LaFarge 05408000
OntarioWI 05407468
Sparta 05382325
LaCrosse_LaCrosseRiver 05383075
Merrill 05395000
MoundsView 05288580
PilotMound 05383950
SaintFrancis 05286000
StCloud 05270500
PineCity 05338500
SpiritFalls 05393500
Danbury 05333500
Fairbault 05353800
Empire 05345000
RainbowLake 05391000
Rothschild 05398000
WisconsinDells 05404000
Babcock 05402000
Ion 05389000
Peever 05290000
Nimrod 05244000
Pillager 05247500
Champlin_ElmCreek 05287890
CannonFalls 05355092
Onamia 05284000
Randolph 05355038
FSHM5 05080000
Homme_Dam 05088500
TraverseRES_Dam-Tailwater 05049710
TraverseRES_Dam 05049700
TraverseWR_Dam 05049995
TraverseWR_Dam-Tailwater 05050000
TraverseWR_Dam-MainLake 05049900
EauGalle_Dam-Tailwater 05370000
LacQuiParle_Dam-T

In [6]:
# you can specify to backfill an individual USGS gage here
usgs_site = '05058000'
cwms_loc = measurement_site_df[measurement_site_df['alias-id']==usgs_site]['location-id'].values[0]

In [7]:
# get the measurements from USGS NWIS
df_meas_usgs, meta = nwis.get_discharge_measurements(sites=usgs_site, channel_rdb_info='1')
df_meas_usgs

,agency_cd,site_no,measurement_nu,measurement_dt,tz_cd,q_meas_used_fg,party_nm,site_visit_coll_agency_cd,gage_height_va,discharge_va,...,chan_area,chan_velocity,chan_stability,chan_material,chan_evenness,long_vel_desc,horz_vel_desc,vert_vel_desc,chan_loc_cd,chan_loc_dist
0,USGS,05058000,1,1949-11-02,NaN,Yes,MCCABE,USGS,26.39,26.0,...,NaN,NaN,UNSP,UNSP,UNSP,unkn,UNSP,UNSP,UNSP,NaN
1,USGS,05058000,2,1949-11-29,NaN,Yes,GABE,USGS,26.41,27.5,...,NaN,NaN,UNSP,UNSP,UNSP,unkn,UNSP,UNSP,UNSP,NaN
2,USGS,05058000,3,1949-11-29,NaN,Yes,GABE,USGS,26.40,26.0,...,NaN,NaN,UNSP,UNSP,UNSP,unkn,UNSP,UNSP,UNSP,NaN
3,USGS,05058000,4,1949-12-30,NaN,Yes,GABE,USGS,26.32,14.0,...,NaN,NaN,UNSP,UNSP,UNSP,unkn,UNSP,UNSP,UNSP,NaN
4,USGS,05058000,5,1950-02-27,NaN,Yes,MCCABE,USGS,26.29,12.0,...,NaN,NaN,UNSP,UNSP,UNSP,unkn,UNSP,UNSP,UNSP,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,USGS,05058000,1013,2025-06-18 11:45:36,CDT,Yes,CSB,USGS,25.17,462.0,...,448.0,1.03,FIRM,CBLS,UNEV,STDY,EVEN,NSTD,DNST,1200.0
1015,USGS,05058000,1014,2025-06-24 15:58:32,CDT,Yes,CMW/CSB,USGS,26.12,790.0,...,612.0,1.29,UNSP,UNSP,UNSP,UNSP,EVEN,UNSP,DNST,1000.0
1016,USGS,05058000,1015,2025-07-16 12:05:49,CDT,Yes,CBL,USGS,25.36,500.0,...,543.0,0.92,UNSP,UNSP,UNSP,UNSP,UNSP,UNSP,UNSP,NaN
1017,USGS,05058000,1016,2025-07-25 09:16:29,CDT,Yes,CBL,USGS,26.83,970.0,...,676.0,1.44,UNSP,UNSP,UNSP,UNSP,UNEV,UNSP,DNST,2700.0


In [8]:
# add cwms location to df_meas_usgs

df_meas_usgs = pd.merge(df_meas_usgs.copy(), usgs_alias_group.df,  how='inner', left_on='site_no', right_on='alias-id')
df_meas_usgs

,agency_cd,site_no,measurement_nu,measurement_dt,tz_cd,q_meas_used_fg,party_nm,site_visit_coll_agency_cd,gage_height_va,discharge_va,...,chan_evenness,long_vel_desc,horz_vel_desc,vert_vel_desc,chan_loc_cd,chan_loc_dist,location-id,office-id,alias-id,attribute
0,USGS,05058000,1,1949-11-02,NaN,Yes,MCCABE,USGS,26.39,26.0,...,UNSP,unkn,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0
1,USGS,05058000,2,1949-11-29,NaN,Yes,GABE,USGS,26.41,27.5,...,UNSP,unkn,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0
2,USGS,05058000,3,1949-11-29,NaN,Yes,GABE,USGS,26.40,26.0,...,UNSP,unkn,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0
3,USGS,05058000,4,1949-12-30,NaN,Yes,GABE,USGS,26.32,14.0,...,UNSP,unkn,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0
4,USGS,05058000,5,1950-02-27,NaN,Yes,MCCABE,USGS,26.29,12.0,...,UNSP,unkn,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,USGS,05058000,1013,2025-06-18 11:45:36,CDT,Yes,CSB,USGS,25.17,462.0,...,UNEV,STDY,EVEN,NSTD,DNST,1200.0,Baldhill_Dam-Tailwater,MVP,05058000,0.0
1015,USGS,05058000,1014,2025-06-24 15:58:32,CDT,Yes,CMW/CSB,USGS,26.12,790.0,...,UNSP,UNSP,EVEN,UNSP,DNST,1000.0,Baldhill_Dam-Tailwater,MVP,05058000,0.0
1016,USGS,05058000,1015,2025-07-16 12:05:49,CDT,Yes,CBL,USGS,25.36,500.0,...,UNSP,UNSP,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0
1017,USGS,05058000,1016,2025-07-25 09:16:29,CDT,Yes,CBL,USGS,26.83,970.0,...,UNSP,UNSP,UNEV,UNSP,DNST,2700.0,Baldhill_Dam-Tailwater,MVP,05058000,0.0


In [9]:
# Define a mapping from USGS tz_cd codes to IANA timezone names
TZ_MAPPING = {
    'AST': 'America/Puerto_Rico',
    'EST': 'America/New York',
    'EDT': 'America/New York',
    'CST': 'America/Chicago',
    'CDT': 'America/Chicago',
    'MST': 'America/Denver',
    'MDT': 'America/Denver',
    'PST': 'America/Los_Angeles',
    'PDT': 'America/Los_Angeles',
    'AKST': 'America/Anchorage',
    'AKDT': 'America/Anchorage',
    'HST': 'Pacific/Honolulu',
    'GST': 'Pacific/Guam',
}
# function to convert the USGS timezone
def convert_to_utc(df, tz_mapping=TZ_MAPPING):
    """
    Converts a pandas DataFrame with timezone-aware datetimes to UTC using a timezone mapping.

    Args:
        df: pandas DataFrame with columns 'measurement_dt' (datetime-like) and 'tz_cd' (timezone code).
        tz_mapping: A dictionary mapping tz_cd codes to IANA timezone names (e.g., {'CST': 'America/Chicago'}).

    Returns:
        pandas DataFrame with an added 'utc_time' column in UTC.  Returns the original dataframe if there is an issue.
    """

    try:
        # Ensure 'measurement_dt' is datetime
        df['measurement_dt'] = pd.to_datetime(df['measurement_dt'], errors='raise', format='ISO8601')  # Raise error for invalid parsing
    except ValueError as e:
        print(f"Error converting 'measurement_dt' to datetime: {e}")
        return df # Return original dataframe

    def to_utc(row):
        dt = row['measurement_dt']
        tz_str = row['tz_cd']

        if pd.isna(tz_str):  # Handle NaNs
            tz = pytz.timezone('UTC')
            dt_aware = tz.localize(dt)
            dt_utc = dt_aware.astimezone(pytz.utc)
            return dt_utc  # Return naive datetime as UTC if no timezone is present

        try:
            iana_tz_name = tz_mapping.get(tz_str)
            if iana_tz_name is None:
                print(f"Unknown timezone code: {tz_str}.  Check your TZ_MAPPING.")
                return dt #return the datetime object if the timezone code is invalid

            tz = pytz.timezone(iana_tz_name)  # Use the IANA name
            if dt.tzinfo is None or dt.tzinfo.utcoffset(dt) is None:  # Check for naive datetime
                dt_aware = tz.localize(dt)
            else:
                dt_aware = dt.astimezone(tz)


            dt_utc = dt_aware.astimezone(pytz.utc)
            return dt_utc
        except pytz.exceptions.UnknownTimeZoneError:
            print(f"Unknown timezone: {tz_str}") # This should rarely happen now
            return dt # Return original datetime if there is a problem with the time zone.
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return dt # Return the original value if an unhandled error occurs


    df['utc_time'] = df.apply(to_utc, axis=1)
    return df





In [10]:
# convert datetime and timezone filed to UTC
df = convert_to_utc(df_meas_usgs.copy()) 
print(df)


     agency_cd   site_no measurement_nu      measurement_dt tz_cd  \
0         USGS  05058000              1 1949-11-02 00:00:00   NaN   
1         USGS  05058000              2 1949-11-29 00:00:00   NaN   
2         USGS  05058000              3 1949-11-29 00:00:00   NaN   
3         USGS  05058000              4 1949-12-30 00:00:00   NaN   
4         USGS  05058000              5 1950-02-27 00:00:00   NaN   
...        ...       ...            ...                 ...   ...   
1014      USGS  05058000           1013 2025-06-18 11:45:36   CDT   
1015      USGS  05058000           1014 2025-06-24 15:58:32   CDT   
1016      USGS  05058000           1015 2025-07-16 12:05:49   CDT   
1017      USGS  05058000           1016 2025-07-25 09:16:29   CDT   
1018      USGS  05058000           1017 2025-08-04 16:37:58   CDT   

     q_meas_used_fg party_nm site_visit_coll_agency_cd  gage_height_va  \
0               Yes   MCCABE                      USGS           26.39   
1               Yes    

In [11]:
# set q_meas_used_fg field as True if Yes and False if No
df.loc[:,'q_meas_used_fg'] = df['q_meas_used_fg'].map({'Yes': True, 'No': False})

# clean up data...find string columns and set NAN to '' and numeric columns to NAN
string_cols = df.select_dtypes(include='object').columns
numeric_cols = df.select_dtypes(include=np.number).columns
df[string_cols] = df[string_cols].astype("string").fillna("")
df[numeric_cols] = df[numeric_cols].fillna(pd.NA) 

# drop rows that don't have a flow or gage height
mask = df[['discharge_va', 'gage_height_va']].isna().all(axis=1)
df = df[~mask]
df






,agency_cd,site_no,measurement_nu,measurement_dt,tz_cd,q_meas_used_fg,party_nm,site_visit_coll_agency_cd,gage_height_va,discharge_va,...,long_vel_desc,horz_vel_desc,vert_vel_desc,chan_loc_cd,chan_loc_dist,location-id,office-id,alias-id,attribute,utc_time
0,USGS,05058000,1,1949-11-02 00:00:00,,True,MCCABE,USGS,26.39,26.0,...,unkn,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-11-02 00:00:00+00:00
1,USGS,05058000,2,1949-11-29 00:00:00,,True,GABE,USGS,26.41,27.5,...,unkn,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-11-29 00:00:00+00:00
2,USGS,05058000,3,1949-11-29 00:00:00,,True,GABE,USGS,26.40,26.0,...,unkn,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-11-29 00:00:00+00:00
3,USGS,05058000,4,1949-12-30 00:00:00,,True,GABE,USGS,26.32,14.0,...,unkn,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-12-30 00:00:00+00:00
4,USGS,05058000,5,1950-02-27 00:00:00,,True,MCCABE,USGS,26.29,12.0,...,unkn,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1950-02-27 00:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,USGS,05058000,1013,2025-06-18 11:45:36,CDT,True,CSB,USGS,25.17,462.0,...,STDY,EVEN,NSTD,DNST,1200.0,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-06-18 16:45:36+00:00
1015,USGS,05058000,1014,2025-06-24 15:58:32,CDT,True,CMW/CSB,USGS,26.12,790.0,...,UNSP,EVEN,UNSP,DNST,1000.0,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-06-24 20:58:32+00:00
1016,USGS,05058000,1015,2025-07-16 12:05:49,CDT,True,CBL,USGS,25.36,500.0,...,UNSP,UNSP,UNSP,UNSP,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-07-16 17:05:49+00:00
1017,USGS,05058000,1016,2025-07-25 09:16:29,CDT,True,CBL,USGS,26.83,970.0,...,UNSP,UNEV,UNSP,DNST,2700.0,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-07-25 14:16:29+00:00


In [12]:
def rename_and_drop_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Renames columns in a pandas DataFrame based on a predefined mapping.
    If a target column name is not provided, the column is dropped.
    Only columns that exist are renamed or dropped.

    Args:
        df: The input pandas DataFrame.

    Returns:
        A new pandas DataFrame with renamed and dropped columns.
    """

    # Define the column mapping
    column_mapping = {
        'agency_cd': 'usgs_agency_cd',
        'site_no': 'usgs_site_no',
        'measurement_nu': 'number',
        'measurement_dt': 'usgs_measurement_dt',
        'tz_cd': 'usgs_tz_cd',
        'q_meas_used_fg': 'used',
        'party_nm': 'party',
        'site_visit_coll_agency_cd': 'agency',
        'discharge_va': 'flow',
        'gage_height_va': 'gage-height',
        'gage_va_change': 'delta-height',
        'gage_va_time': 'delta-time',
        'measured_rating_diff': 'quality',
        'control_type_cd': 'control-condition',
        'discharge_cd': 'flow-adjustment',
        'chan_nu': None,  # Drop this column
        'chan_name': None,  # Drop this column
        'meas_type': None,  # Drop this column
        'streamflow_method': None,  # Drop this column
        'velocity_method': None,  # Drop this column
        'chan_discharge': 'channel-flow',
        'chan_width': 'top-width',
        'chan_velocity': 'avg-velocity',
        'chan_area': 'effective-flow-area',
        'chan_stability': None,  # Drop this column
        'chan_material': None,  # Drop this column
        'chan_evenness': None,  # Drop this column
        'long_vel_desc': None,  # Drop this column
        'horz_vel_desc': None,  # Drop this column
        'vert_vel_desc': None,  # Drop this column
        'chan_loc_cd': None,  # Drop this column
        'chan_loc_dist': None,   # Drop this column
        'location-id': 'name',
        'utc_time': 'instant'
    }

    columns_to_drop = [col for col, target in column_mapping.items() if target is None and col in df.columns]
    df = df.drop(columns=columns_to_drop, errors='ignore')


    # Rename existing columns based on the mapping
    columns_to_rename = {col: target for col, target in column_mapping.items() if target is not None and col in df.columns}
    df = df.rename(columns=columns_to_rename, errors='ignore')
    
    # drop columns that do not have gage-height and flow
    df = df.dropna(subset=['gage-height', 'flow']).copy()
    
    

    return df

In [13]:
# rename columns to match cda fields
df_store = rename_and_drop_columns(df.copy())
df_store

,usgs_agency_cd,usgs_site_no,number,usgs_measurement_dt,usgs_tz_cd,used,party,agency,gage-height,flow,...,flow-adjustment,channel-flow,top-width,effective-flow-area,avg-velocity,name,office-id,alias-id,attribute,instant
0,USGS,05058000,1,1949-11-02 00:00:00,,True,MCCABE,USGS,26.39,26.0,...,NONE,26.0,NaN,NaN,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-11-02 00:00:00+00:00
1,USGS,05058000,2,1949-11-29 00:00:00,,True,GABE,USGS,26.41,27.5,...,NONE,27.5,NaN,NaN,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-11-29 00:00:00+00:00
2,USGS,05058000,3,1949-11-29 00:00:00,,True,GABE,USGS,26.40,26.0,...,NONE,26.0,NaN,NaN,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-11-29 00:00:00+00:00
3,USGS,05058000,4,1949-12-30 00:00:00,,True,GABE,USGS,26.32,14.0,...,NONE,14.0,NaN,NaN,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-12-30 00:00:00+00:00
4,USGS,05058000,5,1950-02-27 00:00:00,,True,MCCABE,USGS,26.29,12.0,...,NONE,12.0,NaN,NaN,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1950-02-27 00:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,USGS,05058000,1013,2025-06-18 11:45:36,CDT,True,CSB,USGS,25.17,462.0,...,NONE,462.0,78.7,448.0,1.03,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-06-18 16:45:36+00:00
1015,USGS,05058000,1014,2025-06-24 15:58:32,CDT,True,CMW/CSB,USGS,26.12,790.0,...,NONE,790.0,88.9,612.0,1.29,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-06-24 20:58:32+00:00
1016,USGS,05058000,1015,2025-07-16 12:05:49,CDT,True,CBL,USGS,25.36,500.0,...,NONE,500.0,87.2,543.0,0.92,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-07-16 17:05:49+00:00
1017,USGS,05058000,1016,2025-07-25 09:16:29,CDT,True,CBL,USGS,26.83,970.0,...,NONE,970.0,83.0,676.0,1.44,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-07-25 14:16:29+00:00


## Check for existing data in the database

If the measurement number existing in the database matches the USGS measurements number or if the collection time is within 5 minutes, it's flagged as a duplicate.

If the attribute in Data Acquisition -> USGS Measurement Group is 1, it will overwrite regardless if there are duplicates. If the attribute is anything else, duplicates will be overwritten.


In [14]:
def check_and_drop_duplicates(df_store, df_existing):
    """
    Checks for duplicates based on "number" and "instant" columns and drops them.

    Args:
        df_renamed: The DataFrame to check for duplicates and modify.
        df_existing: The DataFrame to compare against.

    Returns:
        A tuple containing:
            - df_renamed: The modified DataFrame with duplicates removed.
            - df_rejected_number: DataFrame containing rows rejected due to duplicate "number".
            - df_rejected_instant: DataFrame containing rows rejected due to "instant" within 5 minutes of existing.
    """
    
    df_store = df_store.copy()
    df_existing = df_existing.copy()
    
    
    if not df_existing.empty:
    
        # cast number columns as int, sometimes USGS won't resolve to int...drop those rows
        df_invalid = df_store[pd.to_numeric(df_store['number'], errors='coerce').isna()]
        if not df_invalid.empty:
            print(f"Can't resolve measurement numbers {df_invalid['number'].values} to number. Won't store those measurements")

        # Convert the valid rows to numeric and drop the invalid ones
        df_store['number'] = pd.to_numeric(df_store['number'], errors='coerce')  # Convert to numeric, coercing errors to NaN
        df_store = df_store.dropna(subset=['number'])  # Drop rows where 'number' is NaN

        # Convert the 'number' column to int
        df_store.loc[:, 'number'] = df_store['number'].astype(int)

        # Ensure 'instant' columns are datetime objects
        df_store['instant'] = pd.to_datetime(df_store['instant'])
        df_existing['instant'] = pd.to_datetime(df_existing['instant'])


        # Check for duplicate numbers
        mask_number = df_store['number'].isin(df_existing['number'])
        df_rejected_number = df_store[mask_number].copy()  # Store rejected rows
        df_store = df_store[~mask_number]  # Remove duplicates from df_store


        # Check for instants within 5 minutes

        df_rejected_instant = pd.DataFrame(columns=df_store.columns) # Initialize

        indices_to_drop = [] # Keep track of indices to drop efficiently

        for index, row in df_store.iterrows():
            # Find closest time in df_existing
            closest_time = df_existing['instant'].iloc[(df_existing['instant'] - row['instant']).abs().argsort()[:1]]

            # Check if time difference is within 5 minutes (300 seconds)
            if abs((closest_time.iloc[0] - row['instant']).total_seconds()) <= 300:
                df_rejected_instant = pd.concat([df_rejected_instant, row.to_frame().T])
                indices_to_drop.append(index)

        df_store.loc[~df_store.index.isin(indices_to_drop)]




        return df_store, df_rejected_number, df_rejected_instant
    else:
        return df_store, pd.DataFrame(), pd.DataFrame()



In [15]:
# get existing measurements at site to see any conflicts
try:
    df_existing = cwms.get_measurements(location_id_mask=cwms_loc, office_id=OFFICE).df
except:
    df_existing = pd.DataFrame()




In [16]:
df_store, df_rejected_number, df_rejected_instant = check_and_drop_duplicates(df_store, df_existing)
# the data to store
df_store

Can't resolve measurement numbers <StringArray>
['208A', '499C', '500C']
Length: 3, dtype: string to number. Won't store those measurements


,usgs_agency_cd,usgs_site_no,number,usgs_measurement_dt,usgs_tz_cd,used,party,agency,gage-height,flow,...,flow-adjustment,channel-flow,top-width,effective-flow-area,avg-velocity,name,office-id,alias-id,attribute,instant
0,USGS,05058000,1,1949-11-02 00:00:00,,True,MCCABE,USGS,26.39,26.0,...,NONE,26.0,NaN,NaN,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-11-02 00:00:00+00:00
1,USGS,05058000,2,1949-11-29 00:00:00,,True,GABE,USGS,26.41,27.5,...,NONE,27.5,NaN,NaN,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-11-29 00:00:00+00:00
2,USGS,05058000,3,1949-11-29 00:00:00,,True,GABE,USGS,26.40,26.0,...,NONE,26.0,NaN,NaN,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-11-29 00:00:00+00:00
3,USGS,05058000,4,1949-12-30 00:00:00,,True,GABE,USGS,26.32,14.0,...,NONE,14.0,NaN,NaN,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1949-12-30 00:00:00+00:00
4,USGS,05058000,5,1950-02-27 00:00:00,,True,MCCABE,USGS,26.29,12.0,...,NONE,12.0,NaN,NaN,NaN,Baldhill_Dam-Tailwater,MVP,05058000,0.0,1950-02-27 00:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,USGS,05058000,1013,2025-06-18 11:45:36,CDT,True,CSB,USGS,25.17,462.0,...,NONE,462.0,78.7,448.0,1.03,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-06-18 16:45:36+00:00
1015,USGS,05058000,1014,2025-06-24 15:58:32,CDT,True,CMW/CSB,USGS,26.12,790.0,...,NONE,790.0,88.9,612.0,1.29,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-06-24 20:58:32+00:00
1016,USGS,05058000,1015,2025-07-16 12:05:49,CDT,True,CBL,USGS,25.36,500.0,...,NONE,500.0,87.2,543.0,0.92,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-07-16 17:05:49+00:00
1017,USGS,05058000,1016,2025-07-25 09:16:29,CDT,True,CBL,USGS,26.83,970.0,...,NONE,970.0,83.0,676.0,1.44,Baldhill_Dam-Tailwater,MVP,05058000,0.0,2025-07-25 14:16:29+00:00


In [17]:

cwms_missing_value = -340282346638528859811704183484516925440
# Function to remove NaN values except for specified keys
def remove_nan_values(data, keys_to_keep):
    
    """
    Recursively remove keys with NaN values from a dictionary, except for specified keys.
    """

    if isinstance(data, dict):
        return {k: remove_nan_values(v, keys_to_keep) for k, v in data.items() if k in keys_to_keep or not (v is None or (isinstance(v, float) and math.isnan(v)))}
    return data

# Function to create JSON from DataFrame row

def create_json_from_row(row, OFFICE):
    """
    Transforms a DataFrame row into the specified JSON format.
    """

    try:
        instant_value = pd.to_datetime(row['instant']).isoformat()
    except:
        instant_value = None

    json_data = {
        "height-unit": "ft",
        "flow-unit": "cfs",
        "used": bool(row['used']),
        "agency": str(row['agency']),
        "party": str(row['party']),
        "wm-comments": f"imported from get_usgs_measurements backfill {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%SZ')}",
        "instant": instant_value,
        "id": {
            "office-id": OFFICE,
            "name": str(row['name'])
        },
        "number": str(row['number']),
        "streamflow-measurement": {
            "gage-height": float(row['gage-height']) if not math.isnan(row['gage-height']) else cwms_missing_value,
            "flow": float(row['flow']) if not math.isnan(row['flow']) else cwms_missing_value,
            "quality": str(row['quality'])
        },
        "usgs-measurement": {
            "control-condition": str(row['control-condition']) if not row['control-condition'] =='Unspecifed' else None,
            "flow-adjustment": str(row['flow-adjustment']),
            "delta-height": float(row['delta-height']),
            "delta-time": float(row['delta-time']),
        }
    }

    # Remove keys with NaN values except for 'id', 'number', and 'streamflow-measurement'
    keys_to_keep = ['id', 'number', 'streamflow-measurement']
    json_data = remove_nan_values(json_data, keys_to_keep)
    
    # Remove NaN values from nested 'usgs-measurement' dictionary
    json_data['usgs-measurement'] = {k: v for k, v in json_data['usgs-measurement'].items() if not (v is None or (isinstance(v, float) and math.isnan(v)) or v == 'nan')}

    return json_data




In [18]:
# Iterate through each row of the DataFrame
json_list = []
for index, row in df_store.iterrows():
    json_list.append(create_json_from_row(row, OFFICE))
    # uncomment the following two lines if you need to troubleshoot why a specific measurement is not storing
    # print(f'storing row {row}')
    # cwms.store_measurements(data=[create_json_from_row(row, OFFICE)], fail_if_exists=False)

# # Print the resulting JSON
print(json.dumps(json_list, indent=2))


[
  {
    "height-unit": "ft",
    "flow-unit": "cfs",
    "used": true,
    "agency": "USGS",
    "party": "MCCABE",
    "wm-comments": "imported from get_usgs_measurements backfill 2025-08-11 17:24:27Z",
    "instant": "1949-11-02T00:00:00+00:00",
    "id": {
      "office-id": "MVP",
      "name": "Baldhill_Dam-Tailwater"
    },
    "number": "1",
    "streamflow-measurement": {
      "gage-height": 26.39,
      "flow": 26.0,
      "quality": "Fair"
    },
    "usgs-measurement": {
      "control-condition": "",
      "flow-adjustment": "NONE",
      "delta-height": 0.0,
      "delta-time": 0.6
    }
  },
  {
    "height-unit": "ft",
    "flow-unit": "cfs",
    "used": true,
    "agency": "USGS",
    "party": "GABE",
    "wm-comments": "imported from get_usgs_measurements backfill 2025-08-11 17:24:27Z",
    "instant": "1949-11-29T00:00:00+00:00",
    "id": {
      "office-id": "MVP",
      "name": "Baldhill_Dam-Tailwater"
    },
    "number": "2",
    "streamflow-measurement": {
   

In [ ]:
# store the measurement
cwms.store_measurements(data=json_list, fail_if_exists=True)

In [20]:
# check the data
df_check  = cwms.get_measurements(location_id_mask=cwms_loc, office_id=OFFICE).df
df_check

,id.office-id,id.name,number,instant,streamflow-measurement.gage-height,streamflow-measurement.flow,streamflow-measurement.quality,used,agency,wm-comments,...,usgs-measurement.current-rating,usgs-measurement.control-condition,usgs-measurement.flow-adjustment,usgs-measurement.delta-height,usgs-measurement.delta-time,height-unit,flow-unit,temp-unit,velocity-unit,area-unit
0,MVP,Baldhill_Dam-Tailwater,1,1949-11-02T00:00:00Z,26.39,26.0,Fair,True,USGS,imported from get_usgs_measurements backfill 2...,...,,,NONE,0.00,0.60,ft,cfs,F,fps,ft2
1,MVP,Baldhill_Dam-Tailwater,1.0,1949-11-02T00:00:00Z,26.39,26.0,Fair,True,USGS,imported from get_usgs_measurements backfill 2...,...,,,NONE,0.00,0.60,ft,cfs,F,fps,ft2
2,MVP,Baldhill_Dam-Tailwater,10,1950-05-05T00:00:00Z,31.87,2730.0,Unspecified,True,USGS,imported from get_usgs_measurements backfill 2...,...,,,NONE,0.00,1.30,ft,cfs,F,fps,ft2
3,MVP,Baldhill_Dam-Tailwater,10.0,1950-05-05T00:00:00Z,31.87,2730.0,Unspecified,True,USGS,imported from get_usgs_measurements backfill 2...,...,,,NONE,0.00,1.30,ft,cfs,F,fps,ft2
4,MVP,Baldhill_Dam-Tailwater,100,1954-02-24T00:00:00Z,26.31,60.0,Fair,True,USGS,imported from get_usgs_measurements backfill 2...,...,,,NONE,NaN,NaN,ft,cfs,F,fps,ft2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016,MVP,Baldhill_Dam-Tailwater,997.0,2024-02-07T17:56:00Z,24.11,102.0,Poor,True,USGS,imported from get_usgs_measurements backfill 2...,...,,Clear,NONE,0.01,1.96,ft,cfs,F,fps,ft2
2017,MVP,Baldhill_Dam-Tailwater,998,2024-04-03T14:52:40Z,23.36,33.7,Fair,True,USGS,imported from get_usgs_measurements backfill 2...,...,,Clear,NONE,0.01,1.00,ft,cfs,F,fps,ft2
2018,MVP,Baldhill_Dam-Tailwater,998.0,2024-04-03T14:52:40Z,23.36,33.7,Fair,True,USGS,imported from get_usgs_measurements backfill 2...,...,,Clear,NONE,0.01,1.00,ft,cfs,F,fps,ft2
2019,MVP,Baldhill_Dam-Tailwater,999,2024-04-23T19:25:23Z,24.38,166.0,Good,True,USGS,imported from get_usgs_measurements backfill 2...,...,,Clear,NONE,0.00,1.00,ft,cfs,F,fps,ft2
